# 端到端示例 3：Partition Discovery

目标：在 campaign task 中用多次实验学习分配趋势。每个 final assay 结束一个 experiment，但不结束整个 campaign。

重点：不要假设 hidden partition coefficient 可见；只能从公开仪器和 final assay 推断。

## 1. 任务规划

读取 campaign 状态、可用动作和任务提示。

In [ ]:
from __future__ import annotations

from pathlib import Path

import gymnasium as gym
import pandas as pd
from IPython.display import display

import chemworld  # registers ChemWorld

OUTPUT_DIR = Path("runs/end_to_end")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.precision", 4)

env = gym.make("ChemWorld", task_id="partition-discovery", seed=0)
obs, info = env.reset(seed=0)
display(env.unwrapped.task_prompt())
display(env.unwrapped.campaign_state())
display(pd.DataFrame(env.unwrapped.available_actions()).head(14))
env.close()


## 2. 设计多轮 campaign recipes

这里比较两种 solvent 和两个 target phase。每轮 experiment 都要重新建立相系统。

In [ ]:
experiment_recipes = [
    [
        {"operation": "add_solvent", "volume_L": 0.030, "solvent": 0},
        {"operation": "add_reagent", "amount_mol": 0.008},
        {"operation": "add_phase", "phase": "organic", "volume_L": 0.020},
        {"operation": "add_extractant", "extractant": "organic", "volume_L": 0.010},
        {"operation": "mix", "duration_s": 120.0, "stirring_speed_rpm": 500.0},
        {"operation": "settle", "duration_s": 300.0},
        {"operation": "separate_phase", "target_phase": "organic"},
        {"operation": "terminate"},
        {"operation": "measure", "instrument": "final_assay"},
    ],
    [
        {"operation": "add_solvent", "volume_L": 0.030, "solvent": 3},
        {"operation": "add_reagent", "amount_mol": 0.008},
        {"operation": "add_phase", "phase": "organic", "volume_L": 0.020},
        {"operation": "add_extractant", "extractant": "organic", "volume_L": 0.010},
        {"operation": "mix", "duration_s": 180.0, "stirring_speed_rpm": 600.0},
        {"operation": "settle", "duration_s": 300.0},
        {"operation": "separate_phase", "target_phase": "aqueous"},
        {"operation": "terminate"},
        {"operation": "measure", "instrument": "final_assay"},
    ],
]


## 3. 执行 campaign 并提取每轮 final assay

在 campaign 模式下，`terminated` 通常不会在第一次 final assay 后变成 True；环境会准备下一轮 experiment。

In [ ]:
env = gym.make("ChemWorld", task_id="partition-discovery", seed=0)
obs, info = env.reset(seed=0)
rows = []
final_packets = []
for experiment_index, recipe in enumerate(experiment_recipes):
    for action in recipe:
        validation = env.unwrapped.validate_action(action)
        obs, reward, terminated, truncated, info = env.step(action)
        rows.append({
            "experiment": experiment_index,
            "operation": action["operation"],
            "valid_before_step": validation["valid"],
            "reward": float(reward),
            "leaderboard_score": info.get("leaderboard_score"),
            "precondition_failed": info.get("constraint_flags", {}).get("precondition_failed"),
            "phase_ratio": info.get("processed_estimate", {}).get("phase_ratio"),
            "product_in_organic": info.get("processed_estimate", {}).get("product_in_organic"),
            "product_in_aqueous": info.get("processed_estimate", {}).get("product_in_aqueous"),
            "terminated": terminated,
            "truncated": truncated,
        })
        if action["operation"] == "measure" and action.get("instrument") == "final_assay":
            final_packets.append({"experiment": experiment_index, "info": info})
        if terminated or truncated:
            break
    if terminated or truncated:
        break
df = pd.DataFrame(rows)
display(df)
display(env.unwrapped.campaign_state())
env.close()
df.to_csv(OUTPUT_DIR / "partition_discovery_campaign.csv", index=False)


## 4. 谱图与分配趋势

比较每轮 final assay 的 processed estimates。若 raw spectra 与 processed estimate 冲突，应记录 warning，而不是强行解释。

In [ ]:
def run_recipe(task_id: str, recipe: list[dict], *, seed: int = 0) -> tuple[pd.DataFrame, dict]:
    env = gym.make("ChemWorld", task_id=task_id, seed=seed)
    obs, info = env.reset(seed=seed)
    rows = []
    final_info = info
    try:
        for step, action in enumerate(recipe, start=1):
            validation = env.unwrapped.validate_action(action)
            obs, reward, terminated, truncated, info = env.step(action)
            final_info = info
            rows.append(
                {
                    "step": step,
                    "operation": action["operation"],
                    "valid_before_step": validation["valid"],
                    "invalid_reasons": "; ".join(validation.get("invalid_reasons", [])),
                    "reward": float(reward),
                    "leaderboard_score": info.get("leaderboard_score"),
                    "precondition_failed": info.get("constraint_flags", {}).get("precondition_failed"),
                    "unsafe": info.get("constraint_flags", {}).get("unsafe"),
                    "cost": info.get("cost"),
                    "observed_keys": ", ".join(info.get("observed_keys", [])),
                    "has_raw_signal": bool(info.get("raw_signal")),
                    "terminated": terminated,
                    "truncated": truncated,
                }
            )
            if terminated or truncated:
                break
    finally:
        env.close()
    return pd.DataFrame(rows), final_info


def spectra_frame(final_info: dict, channel: str = "hplc") -> pd.DataFrame:
    packet = final_info.get("raw_signal", {})
    spectra = packet.get("spectra", {}) if isinstance(packet, dict) else {}
    signal = spectra.get(channel, {})
    if channel in {"hplc", "gc"}:
        return pd.DataFrame({"x": signal.get("time_min", []), "signal": signal.get("intensity", [])})
    if channel == "uvvis":
        return pd.DataFrame({"x": signal.get("wavelength_nm", []), "signal": signal.get("absorbance", [])})
    return pd.DataFrame()

summary_rows = []
for packet in final_packets:
    info = packet["info"]
    metrics = info.get("processed_estimate", {})
    summary_rows.append({
        "experiment": packet["experiment"],
        "leaderboard_score": info.get("leaderboard_score"),
        "phase_ratio": metrics.get("phase_ratio"),
        "product_in_organic": metrics.get("product_in_organic"),
        "product_in_aqueous": metrics.get("product_in_aqueous"),
        "has_spectra": bool(info.get("raw_signal", {}).get("spectra")),
    })
summary = pd.DataFrame(summary_rows)
display(summary)
if final_packets:
    hplc = spectra_frame(final_packets[0]["info"], "hplc")
    if not hplc.empty:
        hplc.plot(x="x", y="signal", title="Partition discovery final HPLC", xlabel="time / min")


## 5. 反思记录

请写出你当前学到的局部分配模型，以及下一轮应该改哪个变量。

In [ ]:
reflection = {
    "observed_pattern": summary.to_dict(orient="records"),
    "local_world_model": "示例：当前 public observations 只能支持相对分配趋势，不能直接给出 hidden partition coefficient。",
    "next_experiment": {"variable": "solvent / target_phase", "reason": "验证 organic 与 aqueous readout 的稳定差异"},
    "evidence": str(OUTPUT_DIR / "partition_discovery_campaign.csv"),
}
reflection
